# PostgreSQL Cheat Sheet
**DDL · DML · DCL · DQL**  
*A quick reference for PostgreSQL syntax.*

---

## 🗂️ DDL – Data Definition Language
*Schema & structure commands.*
```sql

--CREATE DATABASE
CREATE DATABASE mydb;

--DROP DATABASE
DROP DATABASE IF EXISTS mydb;

--CREATE TABLE
CREATE TABLE users ( id SERIAL PRIMARY KEY, name TEXT NOT NULL, email TEXT UNIQUE, created_at TIMESTAMPTZ DEFAULT now() );

--GENERATED AS IDENTITY (preferred over SERIAL) 
id INT GENERATED ALWAYS AS IDENTITY 

-- ALTER TABLE
ALTER TABLE users ADD COLUMN age INT;
ALTER TABLE users DROP COLUMN age;
ALTER TABLE users ALTER COLUMN name SET NOT NULL;
ALTER TABLE users RENAME COLUMN email TO mail;

--DROP TABLE
DROP TABLE IF EXISTS users CASCADE;

--TRUNCATE 
TRUNCATE TABLE users RESTART IDENTITY CASCADE;

--CREATE INDEX
CREATE INDEX idx_users_email ON users (email);
CREATE UNIQUE INDEX ON users (lower(email));
CREATE INDEX ON items USING GIN (tags); --GIN index

-- Constraints
PRIMARY KEY, FOREIGN KEY ... REFERENCES, UNIQUE, CHECK (age >= 0), NOT NULL

-- CREATE SCHEMA
CREATE SCHEMA accounting;

-- CREATE EXTENSION
CREATE EXTENSION IF NOT EXISTS "uuid-ossp";

-- ALTER SEQUENCE
ALTER SEQUENCE users_id_seq RESTART WITH 100;
```
---

## ✏️ DML – Data Manipulation Language
*Insert, update, delete data.*
```sql

-- INSERT
INSERT INTO users (name, email) VALUES ('Alice', 'alice@example.com'); 

INSERT INTO users (name, email) VALUES ('Bob','bob@x.com') RETURNING id;

--INSERT … SELECT
INSERT INTO archived_users SELECT * FROM users WHERE active = false;

-- UPSERT
INSERT INTO users (email, name) VALUES ('a@b.com','Zoe') ON CONFLICT (email) DO UPDATE SET name = EXCLUDED.name RETURNING *;

--MERGE
MERGE INTO account AS a USING transactions AS t ON a.id = t.acc_id WHEN MATCHED THEN UPDATE SET balance = balance + t.amount WHEN NOT MATCHED THEN INSERT (id, balance) VALUES (t.acc_id, t.amount);

--UPDATE
UPDATE users SET name = 'Bob' WHERE id = 2 RETURNING *;

`Update with FROM:` UPDATE users SET city = c.name FROM cities c WHERE users.city_id = c.id;

--DELETE
DELETE FROM users WHERE last_login < now() - INTERVAL '1 year';

`Using USING:` DELETE FROM users USING addresses a WHERE users.id = a.user_id AND a.city IS NULL;

--RETURNING clause

`Append` RETURNING * or RETURNING id, name `to INSERT/UPDATE/DELETE to return affected rows`. 
```
---

## 🔐 DCL – Data Control Language
*Permissions, roles, privileges.*
```sql

--GRANT
GRANT SELECT, INSERT ON users TO analyst;` <br> `GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA public TO admin;` <br> `GRANT USAGE ON SCHEMA accounting TO billing_team;

--REVOKE
REVOKE DELETE ON users FROM analyst;

--ROLES & USERS
CREATE ROLE analyst WITH LOGIN PASSWORD 'secret';
CREATE ROLE superuser WITH LOGIN SUPERUSER;
ALTER ROLE analyst CREATEDB;
DROP ROLE analyst;

--Row-Level Security
ALTER TABLE orders ENABLE ROW LEVEL SECURITY;
CREATE POLICY user_orders ON orders FOR SELECT USING (user_id = current_user_id());

--Default Privileges
ALTER DEFAULT PRIVILEGES IN SCHEMA public GRANT SELECT ON TABLES TO read_only;
```
---

## 🔍 DQL – Data Query Language
*The SELECT statement and its clauses.*

### Basic Query Order
```sql
SELECT [DISTINCT] columns / aggregates / window functions
FROM table
[ JOIN … ON … ]
[ WHERE conditions ]
[ GROUP BY columns ]
[ HAVING aggregate_condition ]
[ ORDER BY columns [ASC|DESC] ]
[ LIMIT number OFFSET number ];
```

### Common Clauses & Examples
```sql

--SELECT
SELECT name, age FROM users;

--DISTINCT
SELECT DISTINCT city FROM users;

-- WHERE
WHERE age >= 18 AND status = 'active';
WHERE created_at BETWEEN '2024-01-01' AND now();` <br> `WHERE email LIKE '%@gmail.com';

--GROUP BY / HAVING
SELECT city, COUNT(*) FROM users GROUP BY city HAVING COUNT(*) > 5;

--ORDER BY
ORDER BY created_at DESC, name ASC NULLS LAST;

--LIMIT / OFFSET
LIMIT 10 OFFSET 20; 
```
### Joins
```sql
-- INNER JOIN (default JOIN)
SELECT u.name, o.total FROM users u JOIN orders o ON u.id = o.user_id;

-- LEFT / RIGHT / FULL OUTER JOIN
SELECT u.name, o.id FROM users u LEFT JOIN orders o ON u.id = o.user_id;

-- CROSS JOIN
SELECT * FROM colors CROSS JOIN sizes;

-- LATERAL join
SELECT u.name, latest.* FROM users u
CROSS JOIN LATERAL (SELECT * FROM orders WHERE user_id = u.id ORDER BY created_at DESC LIMIT 1) latest;
```

### Subqueries & CTEs
```sql
-- Subquery in WHERE
SELECT name FROM users WHERE id IN (SELECT user_id FROM orders WHERE total > 100);

-- EXISTS
SELECT * FROM users u WHERE EXISTS (SELECT 1 FROM orders o WHERE o.user_id = u.id);

-- Common Table Expression (WITH)
WITH recent_users AS (
    SELECT * FROM users WHERE created_at > now() - INTERVAL '7 days'
)
SELECT city, COUNT(*) FROM recent_users GROUP BY city;

-- Recursive CTE
WITH RECURSIVE subordinates AS (
    SELECT id, manager_id, name FROM employees WHERE id = 1
    UNION ALL
    SELECT e.id, e.manager_id, e.name FROM employees e
    INNER JOIN subordinates s ON e.manager_id = s.id
) SELECT * FROM subordinates;
```

### Window Functions
```sql
SELECT
    name,
    department,
    salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank,
    SUM(salary) OVER (PARTITION BY department) AS dept_total,
    AVG(salary) OVER (ORDER BY salary ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING) AS moving_avg
FROM employees;
```

### Useful Functions & Operators
```sql

--String : ('Hello' \|\| ' ' \|\| 'World')
length(name), lower(), substring(), replace()

--Date/Time
now(), CURRENT_DATE, age(timestamp), 
date_trunc('month', created_at), 
EXTRACT(year FROM dob)

--Conditional
COALESCE(phone, 'No phone'), 
NULLIF(val, 0), 
CASE WHEN age < 18 THEN 'child' ELSE 'adult' END

--Casting
'123'::INT, 
CAST('2024-01-01' AS DATE)

--JSON
data->'key', 
data->>'key' (text), 
jsonb_array_elements(arr), 
@> `containment operator` 

-- Arrays
ARRAY[1,2,3], 
tags @> ARRAY['urgent'], 
unnest(array_column)

--Math/Stats
AVG(), COUNT(), SUM(), MAX(), MIN(), 
PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY x)
```
---

### 💡 Bonus: TCL – Transaction Control
*(Often used together with DML)*  
```sql
BEGIN;          -- start transaction
-- DML statements
SAVEPOINT sp1;
ROLLBACK TO sp1;
COMMIT;         -- save changes
ROLLBACK;       -- discard changes
```

> **Tip:** In PostgreSQL, most DDL commands can be rolled back if wrapped in a transaction.

If you want to search **both nested `JSONB` fields** and **`pgvector` embeddings** together in PostgreSQL, the usual production pattern is:

* Store metadata/features in `JSONB`
* Store embeddings in `vector`
* Use:

  * `GIN` index for JSONB
  * `HNSW` or `IVFFlat` index for vectors
* Combine:

  * structured filtering (`WHERE`)
  * semantic similarity (`ORDER BY embedding <-> query_vector`)

This is extremely common in:

* retail search
* RAG systems
* recommendation engines
* catalog search
* AI agents with metadata filtering

---

# 1. Example Table Design

```sql
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE products (
    id BIGSERIAL PRIMARY KEY,

    title TEXT,

    metadata JSONB,

    embedding VECTOR(768)
);
```

---

# 2. Example JSONB Structure

```json
{
  "brand": "Nike",
  "category": {
    "main": "Shoes",
    "sub": "Running"
  },
  "price": 4999,
  "attributes": {
    "color": "black",
    "size": 10
  },
  "tags": ["sports", "running", "men"]
}
```

---

# 3. Important Indexes

## JSONB GIN Index

```sql
CREATE INDEX idx_products_metadata
ON products
USING GIN(metadata);
```

This accelerates:

* containment
* key existence
* nested lookups

---

## pgvector HNSW Index

```sql
CREATE INDEX idx_products_embedding
ON products
USING hnsw (embedding vector_cosine_ops);
```

Or IVFFlat:

```sql
CREATE INDEX idx_products_embedding_ivf
ON products
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);
```

---

# 4. Search Nested JSONB Fields

---

## A. Exact Nested Match

### Find all running shoes

```sql
SELECT *
FROM products
WHERE metadata->'category'->>'sub' = 'Running';
```

---

## B. Numeric Filter

```sql
SELECT *
FROM products
WHERE (metadata->>'price')::INT < 6000;
```

---

## C. Nested Attribute Search

```sql
SELECT *
FROM products
WHERE metadata->'attributes'->>'color' = 'black';
```

---

## D. Array Contains

```sql
SELECT *
FROM products
WHERE metadata->'tags' ? 'sports';
```

OR

```sql
SELECT *
FROM products
WHERE metadata @> '{"tags":["sports"]}';
```

---

# 5. Combine JSONB + pgvector Search

This is the real-world AI retrieval pattern.

---

## Example:

### "Find black Nike running shoes semantically similar to query"

```sql
SELECT
    id,
    title,
    metadata,
    embedding <=> '[0.12, 0.44, ...]' AS distance
FROM products
WHERE
    metadata->>'brand' = 'Nike'
    AND metadata->'category'->>'sub' = 'Running'
    AND metadata->'attributes'->>'color' = 'black'
ORDER BY embedding <=> '[0.12, 0.44, ...]'
LIMIT 10;
```

---

# 6. Better Production Pattern (Pre-filter → Vector Search)

Vector search is expensive.

Best practice:

```sql
WITH filtered AS (
    SELECT *
    FROM products
    WHERE
        metadata->>'brand' = 'Nike'
        AND metadata->'category'->>'sub' = 'Running'
)
SELECT
    *,
    embedding <=> '[0.12, 0.44, ...]' AS score
FROM filtered
ORDER BY embedding <=> '[0.12, 0.44, ...]'
LIMIT 10;
```

Why?

* Reduces vector comparisons
* Much faster at scale
* Better recall/latency balance

---

# 7. Hybrid Search (BM25 + JSONB + Vector)

Very common in retail AI systems.

```sql
SELECT
    id,
    title,
    ts_rank(search_tsv, query) * 0.3 +
    (1 - (embedding <=> query_embedding)) * 0.7 AS final_score
FROM products,
     plainto_tsquery('running shoes') query,
     (
       SELECT '[0.12, 0.44, ...]'::vector AS query_embedding
     ) q
WHERE
    search_tsv @@ query
    AND metadata->>'brand' = 'Nike'
ORDER BY final_score DESC
LIMIT 20;
```

This gives:

* keyword relevance
* semantic relevance
* metadata filtering

Exactly what large retail/search systems do.

---

# 8. Fast Nested JSONB Search with Expression Indexes

If a nested field is queried frequently:

```sql
CREATE INDEX idx_brand
ON products ((metadata->>'brand'));
```

Nested:

```sql
CREATE INDEX idx_subcategory
ON products ((metadata->'category'->>'sub'));
```

This is MUCH faster than generic GIN for hot filters.

---

# 9. Best Production Architecture

Typical enterprise retrieval pipeline:

```text
User Query
    ↓
LLM Intent Extraction
    ↓
Structured Filters
(brand/category/price)
    ↓
JSONB Pre-filter
    ↓
Vector Similarity Search
(pgvector)
    ↓
Hybrid Ranker
    ↓
Final Results
```

This is how:

* Amazon-style retrieval
* AI retail search
* RAG with metadata
* recommendation engines
  work internally.

---

# 10. Advanced: JSONPath Queries

PostgreSQL supports JSONPath.

---

## Example

```sql
SELECT *
FROM products
WHERE metadata @? '$.attributes.color == "black"';
```

Nested:

```sql
SELECT *
FROM products
WHERE metadata @? '$.category.sub == "Running"';
```

Useful for:

* dynamic filter builders
* agentic querying
* arbitrary nested schemas

---

# 11. Real AI Retail Example

Imagine query:

> "cheap black running shoes for men"

Your system might:

---

## LLM extracts:

```json
{
  "category": "Running",
  "color": "black",
  "gender": "men",
  "max_price": 5000
}
```

---

## SQL

```sql
SELECT
    id,
    title,
    embedding <=> :query_embedding AS distance
FROM products
WHERE
    metadata->'category'->>'sub' = 'Running'
    AND metadata->'attributes'->>'color' = 'black'
    AND metadata->>'gender' = 'men'
    AND (metadata->>'price')::INT <= 5000
ORDER BY embedding <=> :query_embedding
LIMIT 20;
```

This is the exact retrieval pattern used in modern AI commerce systems.

---

# 12. Important Performance Advice

## DO NOT

```sql
ORDER BY embedding <=> query_vector
```

on the entire table blindly.

---

## ALWAYS

1. Metadata filter first
2. Reduce candidate set
3. Then vector search

Otherwise latency explodes at scale.

---

# 13. Recommended Index Strategy

| Use Case            | Index            |
| ------------------- | ---------------- |
| General JSON search | GIN(jsonb)       |
| Frequent nested key | Expression index |
| Vector ANN          | HNSW             |
| Huge datasets       | IVFFlat          |
| Text search         | GIN(tsvector)    |

---

# 14. Enterprise-Level Optimization

At large scale:

```text
JSONB Filter
    ↓
Candidate IDs
    ↓
Approx ANN Search
    ↓
Cross Encoder Re-rank
    ↓
Business Rules
```

Common stack:

* PostgreSQL
* pgvector
* Redis cache
* reranker model
* FastAPI retrieval layer

Very similar to the hybrid RL-based retail retrieval system you were discussing earlier.
